In [2]:
# Image scraper

# TODO
# [X] try downloading a single image from a given url and see if it works
# [ ] make this script work to get image url
# then:
# [ ] make big loopy script that adds the image url to the xlsx sheet
# [ ] IF ALL WORKS: maker another script to actually download images from the excel sheets

# 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from datetime import datetime
import calendar
import time
import requests


# Helper functions

def _parse_date(date_str):
    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%Y/%m/%d"):
        try:
            return datetime.strptime(date_str, fmt)
        except Exception:
            pass
    raise ValueError("date must be MM/DD/YYYY or YYYY-MM-DD")

def _safe_click(driver, element):
    try:
        element.click()
    except Exception:
        driver.execute_script("arguments[0].click();", element)

        
def get_image_url_for_timestamp(camera_name, date_str, time_str, headless=True, timeout=20):
    """
    camera_name: "NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte"
    date_str: "08/01/2025" or "2025-08-01"
    time_str: "HH:MM"
    Returns dict with {"img_url": ..., "download_href": ...}
    """
    dt = _parse_date(date_str)
    hour, minute_target = map(int, time_str.split(":"))

    url = f"https://apps.usgs.gov/hivis/camera/{camera_name}"

    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=opts)
    wait = WebDriverWait(driver, timeout)

    try:
        driver.get(url)

        src_before = None
        try:
            img_before = driver.find_element(By.CSS_SELECTOR, "img[alt='Displayed']")
            src_before = img_before.get_attribute("src")
        except Exception:
            pass

        # Calendar icon
        cal_btn = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[.//svg[@data-testid='CalendarMonthIcon']]"))
        )
        _safe_click(driver, cal_btn)

        wait.until(EC.presence_of_element_located((By.XPATH, "//h6[normalize-space()='Year']")))

        def click_under_heading(heading, text):
            xpath = f"//h6[normalize-space()='{heading}']/following::div[1]//button[normalize-space()='{text}']"
            try:
                btn = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
                _safe_click(driver, btn)
                time.sleep(0.2)
                return True
            except TimeoutException:
                try:
                    btn2 = wait.until(EC.element_to_be_clickable((By.XPATH, f"//button[normalize-space()='{text}']")))
                    _safe_click(driver, btn2)
                    time.sleep(0.2)
                    return True
                except TimeoutException:
                    return False

        # Year
        if not click_under_heading("Year", str(dt.year)):
            raise RuntimeError("Year not clickable")

        # Month
        if not click_under_heading("Month", calendar.month_abbr[dt.month]):
            if not click_under_heading("Month", calendar.month_name[dt.month]):
                raise RuntimeError("Month not clickable")

        # Day
        if not click_under_heading("Day", str(dt.day)):
            raise RuntimeError("Day not clickable")

        # Hour
        if not click_under_heading("Hour", str(hour)):
            if not click_under_heading("Hour", f"{hour:02d}"):
                raise RuntimeError("Hour not clickable")

        # Minute (choose nearest option)
        minute_buttons = driver.find_elements(By.XPATH, "//h6[normalize-space()='Minute']/following::div[1]//button")
        if minute_buttons:
            candidates = [(int(b.text.strip()), b) for b in minute_buttons if b.text.strip().isdigit()]
            if candidates:
                best_val, best_elem = min(candidates, key=lambda vb: abs(vb[0] - minute_target))
                _safe_click(driver, best_elem)
        else:
            click_under_heading("Minute", str(minute_target))

        # Wait for image update
        start = time.time()
        new_src = None
        while time.time() - start < timeout:
            try:
                img = driver.find_element(By.CSS_SELECTOR, "img[alt='Displayed']")
                cur = img.get_attribute("src")
                if cur and cur != src_before:
                    new_src = cur
                    break
            except Exception:
                pass
            time.sleep(0.3)

        # Optional download link
        download_href = None
        try:
            dl_btn = driver.find_element(By.XPATH, "//button[.//svg[@data-testid='DownloadIcon']]")
            _safe_click(driver, dl_btn)
            time.sleep(0.5)
            a_elem = driver.find_element(By.CSS_SELECTOR, "a[download]")
            download_href = a_elem.get_attribute("href")
        except Exception:
            pass

        return {"img_url": new_src, "download_href": download_href}

    finally:
        driver.quit()

        
def download_url_to_file(url, out_path=None):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    if out_path is None:
        out_path = url.split("/")[-1]
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return out_path


# Example usage

CAMERA = "NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte"
DATE = "08/01/2025"
TIME = "04:10"

out = get_image_url_for_timestamp(CAMERA, DATE, TIME, headless=False, timeout=30)
print("IMG URL:", out["img_url"])
print("DOWNLOAD HREF:", out["download_href"])

if out["img_url"]:
    saved_path = download_url_to_file(out["img_url"])
    print("Saved to", saved_path)


ModuleNotFoundError: No module named 'selenium'

In [7]:
# 4. Download it
#img_url = "https://apps.usgs.gov/hivis/camera/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte#&gid=hivis&pid=NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte___2025-08-01T04-10-22Z.jpg"
#img_url = "https://usgs-nims-images.s3.amazonaws.com/overlay/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte___2025-08-01T04-10-22Z.jpg"
img_url = "https://usgs-nims-images.s3.amazonaws.com/720/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte___2025-09-30T02-25-23Z.jpg"
r = requests.get(img_url)
with open(img_url.split("/")[-1], "wb") as f:
    f.write(r.content)

print("Saved", img_url.split("/")[-1])

Saved NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte___2025-09-30T02-25-23Z.jpg
